# 02 — Classical Machine Learning Baselines & The Accuracy Paradox (Part C)

### Models Evaluated
1. **Naive Majority Baseline** (predicts `0` / Legitimate for every transaction — demonstrates the **Accuracy Paradox**)
2. **Static Rule Engine** (`isFlaggedFraud`)
3. **Logistic Regression** (`class_weight='balanced'`)
4. **Isolation Forest** (unsupervised tree-based anomaly detector fit strictly on legitimate training rows)
5. **XGBoost + SMOTE** (Synthetic Minority Over-sampling applied **strictly to the training split** to avoid data leakage)

In [1]:
import json
from pathlib import Path
import pandas as pd
import joblib

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ml_metrics = json.loads((ROOT / 'models' / 'ml_metrics.json').read_text())
print('SMOTE Training Split Before:', ml_metrics['smote_before'])
print('SMOTE Training Split After :', ml_metrics['smote_after'])

df_ml = pd.DataFrame(ml_metrics['ml_results'])[
    ['model', 'family', 'accuracy', 'pr_auc', 'f1', 'precision', 'recall', 'tp', 'fp', 'fn', 'tn', 'latency_ms']
]
df_ml

SMOTE Training Split Before: {'total': 76245, 'fraud': 828}
SMOTE Training Split After : {'total': 86729, 'fraud': 11312}


,model,family,accuracy,pr_auc,f1,precision,recall,tp,fp,fn,tn,latency_ms
0,Naive Majority Baseline (All Legit),Baseline,0.989045,0.010955,0.000000,0.000000,0.000000,0,0,179,16160,0.001000
1,Static Rule Engine (isFlaggedFraud),Rule-Based,0.989045,0.010955,0.000000,0.000000,0.000000,0,0,179,16160,0.005000
2,Logistic Regression,Classical ML,0.997674,0.928877,0.881250,1.000000,0.787709,141,0,38,16160,0.000151
3,Isolation Forest,Unsupervised ML,0.989779,0.687753,0.591687,0.526087,0.675978,121,109,58,16051,0.033788
4,XGBoost + SMOTE,Ensemble ML,0.997797,0.939784,0.888199,1.000000,0.798883,143,0,36,16160,0.003653


## Why Raw Accuracy is Misleading on Imbalanced Streams
- Notice that the **Naive Majority Baseline** achieves **98.90% accuracy** on the high-risk test stream (and **>99.5% accuracy** across all transaction types) while catching **0 frauds (`TP = 0`, `F1 = 0.0000`, `PR-AUC = 0.0110`)**.
- By contrast, **XGBoost + SMOTE** achieves **PR-AUC = 0.9398** and **F1 = 0.8882**, catching nearly 80% of all fraudulent transactions with zero false alarms at its validation-optimal F1 threshold.

In [2]:
ablation = pd.DataFrame([ml_metrics['ablation_no_balance_error']])[
    ['model', 'accuracy', 'pr_auc', 'f1', 'precision', 'recall']
]
print('Ablation Study (Removing errorBalanceOrig & errorBalanceDest):')
ablation

Ablation Study (Removing errorBalanceOrig & errorBalanceDest):


,model,accuracy,pr_auc,f1,precision,recall
0,XGBoost + SMOTE (No Balance-Error Features),0.997674,0.929167,0.886228,0.954839,0.826816
